# Analýza datových sad – Streaming DB

**Projekt:** NoSQL semestrální práce – MongoDB  
**Datasety:**
- `netflix_titles.csv` – katalog Netflix (8 807 záznamů) – zdroj: https://www.kaggle.com/datasets/shivamb/netflix-shows
- `platform_financials_comprehensive.csv` – finanční metriky platforem (10 záznamů) – zdroj: https://www.kaggle.com/datasets/samyakrajbayar/streaming-platform-comprehensive-dataset
- `streaming_platform_shifts_2026.csv` – tituly na streamovacích platformách v roce 2026 (81 záznamů) – zdroj: https://www.kaggle.com/datasets/samyakrajbayar/streaming-platform-comprehensive-dataset

Všechny tři datasety sdílejí téma **streamovacích platforem** a jsou navzájem propojitelné přes název platformy nebo název titulu.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print('Knihovny načteny.')

## 1. Načtení dat

In [ ]:
# Načtení všech tří datasetů
netflix   = pd.read_csv('netflix_titles.csv')
financials = pd.read_csv('platform_financials_comprehensive.csv')
shifts     = pd.read_csv('streaming_platform_shifts_2026.csv')

print(f'netflix_titles:               {netflix.shape[0]:>6} řádků, {netflix.shape[1]} sloupců')
print(f'platform_financials:          {financials.shape[0]:>6} řádků, {financials.shape[1]} sloupců')
print(f'streaming_platform_shifts:    {shifts.shape[0]:>6} řádků, {shifts.shape[1]} sloupců')

## 2. Základní přehled – struktura dat

In [ ]:
print('=== netflix_titles – prvních 5 řádků ===')
display(netflix.head())

In [ ]:
print('=== platform_financials ===')
display(financials[['platform','subscribers_millions','quarterly_revenue_usd_millions',
                     'quarterly_profit_usd_millions','annual_content_spend_usd_millions']].head(10))

In [ ]:
print('=== streaming_shifts_2026 ===')
display(shifts.head())

In [ ]:
print('=== Datové typy – netflix_titles ===')
display(netflix.dtypes.to_frame('dtype'))

## 3. Analýza chybějících hodnot

In [ ]:
def missing_summary(df, name):
    total   = len(df)
    missing = df.isnull().sum()
    pct     = (missing / total * 100).round(2)
    result  = pd.DataFrame({'chybí': missing, 'procent': pct, 'vyplněno': total - missing})
    result  = result[result['chybí'] > 0].sort_values('chybí', ascending=False)
    print(f'\n=== {name} – chybějící hodnoty (celkem {total} řádků) ===')
    display(result)
    return result

miss_netflix    = missing_summary(netflix,    'netflix_titles')
miss_financials = missing_summary(financials, 'platform_financials')
miss_shifts     = missing_summary(shifts,     'streaming_shifts')

In [ ]:
# Vizualizace chybějících hodnot v netflix_titles
fig, ax = plt.subplots(figsize=(10, 4))
miss_netflix['procent'].sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('% chybějících hodnot')
ax.set_title('Chybějící hodnoty v netflix_titles (%)')
ax.axvline(x=10, color='red', linestyle='--', alpha=0.5, label='10 %')
ax.legend()
plt.tight_layout()
plt.savefig('missing_values.png', dpi=150)
plt.show()

## 4. Základní statistiky – netflix_titles

In [ ]:
print('=== Numerické statistiky ===')
display(netflix[['release_year']].describe().round(1))

print('\n=== Rozložení typů obsahu ===')
display(netflix['type'].value_counts().to_frame('počet'))

In [ ]:
# Rozložení Movie vs TV Show
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = netflix['type'].value_counts()
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=['#4878CF','#6ACC65'], startangle=90)
axes[0].set_title('Poměr filmů a seriálů')

netflix.groupby(['type', netflix['release_year'] // 10 * 10]).size().unstack(0).plot(
    kind='bar', ax=axes[1], color=['#4878CF','#6ACC65'])
axes[1].set_xlabel('Dekáda vydání')
axes[1].set_ylabel('Počet titulů')
axes[1].set_title('Počet titulů podle dekády a typu')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('type_distribution.png', dpi=150)
plt.show()

In [ ]:
# Top 15 zemí
top_countries = (netflix['country']
    .str.split(', ').explode()
    .str.strip()
    .value_counts()
    .head(15))

fig, ax = plt.subplots(figsize=(12, 5))
top_countries.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Top 15 zemí podle počtu titulů v Netflix katalogu')
ax.set_xlabel('Země')
ax.set_ylabel('Počet titulů')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('top_countries.png', dpi=150)
plt.show()

print(top_countries)

In [ ]:
# Distribuce roku vydání
fig, ax = plt.subplots(figsize=(12, 4))
netflix['release_year'].dropna().hist(bins=40, ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Distribuce roku vydání (netflix_titles)')
ax.set_xlabel('Rok vydání')
ax.set_ylabel('Počet titulů')
ax.axvline(netflix['release_year'].median(), color='red', linestyle='--',
           label=f'Medián: {int(netflix["release_year"].median())}')
ax.legend()
plt.tight_layout()
plt.savefig('release_year_dist.png', dpi=150)
plt.show()

In [ ]:
# Distribuce hodnocení
valid_ratings = ['G','PG','PG-13','R','NC-17','NR',
                 'TV-Y','TV-Y7','TV-Y7-FV','TV-G','TV-PG','TV-14','TV-MA']

rating_counts = (netflix[netflix['rating'].isin(valid_ratings)]
    ['rating'].value_counts()
    .sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(12, 4))
rating_counts.plot(kind='bar', ax=ax, color='mediumpurple')
ax.set_title('Distribuce věkových hodnocení (netflix_titles)')
ax.set_xlabel('Hodnocení')
ax.set_ylabel('Počet titulů')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('rating_distribution.png', dpi=150)
plt.show()

print(rating_counts)

In [ ]:
# Délka filmů (histogram)
movies = netflix[netflix['type'] == 'Movie'].copy()
movies['duration_min'] = pd.to_numeric(
    movies['duration'].str.extract(r'(\d+)')[0], errors='coerce')
movies = movies.dropna(subset=['duration_min'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

movies['duration_min'].hist(bins=50, ax=axes[0], color='#6ACC65', edgecolor='white')
axes[0].set_title('Distribuce délky filmů (minuty)')
axes[0].set_xlabel('Délka (min)')
axes[0].set_ylabel('Počet filmů')
axes[0].axvline(movies['duration_min'].mean(), color='red', linestyle='--',
                label=f'Průměr: {movies["duration_min"].mean():.0f} min')
axes[0].legend()

# Počet sezón seriálů
shows = netflix[netflix['type'] == 'TV Show'].copy()
shows['seasons'] = pd.to_numeric(
    shows['duration'].str.extract(r'(\d+)')[0], errors='coerce')
shows['seasons'].value_counts().sort_index().head(15).plot(
    kind='bar', ax=axes[1], color='#4878CF')
axes[1].set_title('Distribuce počtu sezón (TV Show)')
axes[1].set_xlabel('Počet sezón')
axes[1].set_ylabel('Počet seriálů')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('duration_seasons.png', dpi=150)
plt.show()

print(f'Průměrná délka filmu:   {movies["duration_min"].mean():.1f} min')
print(f'Medián délky filmu:     {movies["duration_min"].median():.1f} min')
print(f'Nejkratší film:         {movies["duration_min"].min():.0f} min')
print(f'Nejdelší film:          {movies["duration_min"].max():.0f} min')

In [ ]:
# Trend přidávání obsahu na Netflix
netflix['year_added'] = netflix['date_added'].str.extract(r'(\d{4})$').astype(float)
trend = netflix.groupby(['year_added','type']).size().unstack(fill_value=0)
trend = trend[trend.index >= 2015]

fig, ax = plt.subplots(figsize=(12, 4))
trend.plot(kind='bar', ax=ax, color=['#4878CF','#6ACC65'])
ax.set_title('Počet titulů přidaných na Netflix podle roku')
ax.set_xlabel('Rok přidání')
ax.set_ylabel('Počet titulů')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Typ')
plt.tight_layout()
plt.savefig('content_added_trend.png', dpi=150)
plt.show()

print(trend)

In [ ]:
# Top 10 žánrů
genres = (netflix['listed_in']
    .dropna()
    .str.split(', ')
    .explode()
    .str.strip()
    .value_counts()
    .head(10))

fig, ax = plt.subplots(figsize=(12, 4))
genres.plot(kind='barh', ax=ax, color='coral')
ax.set_title('Top 10 nejčastějších žánrů v Netflix katalogu')
ax.set_xlabel('Počet titulů')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('top_genres.png', dpi=150)
plt.show()

print(genres)

## 5. Analýza – platform_financials

In [ ]:
print('=== Statistiky finančních dat ===')
num_cols = ['subscribers_millions','quarterly_revenue_usd_millions',
            'quarterly_profit_usd_millions','annual_content_spend_usd_millions']
display(financials[['platform'] + num_cols].set_index('platform'))

In [ ]:
# Porovnání platforem – předplatitelé a příjmy
fin_plot = financials.dropna(subset=['subscribers_millions','quarterly_revenue_usd_millions'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fin_plot.set_index('platform')['subscribers_millions'].sort_values().plot(
    kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Počet předplatitelů (mil.)')
axes[0].set_xlabel('Předplatitelé (mil.)')

fin_plot.set_index('platform')['quarterly_revenue_usd_millions'].sort_values().plot(
    kind='barh', ax=axes[1], color='#6ACC65')
axes[1].set_title('Čtvrtletní příjmy (mil. USD)')
axes[1].set_xlabel('Příjmy (mil. USD)')

plt.suptitle('Porovnání streamovacích platforem', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig('platform_comparison.png', dpi=150)
plt.show()

In [ ]:
# Zisková marže platforem
fin_margin = financials.dropna(subset=['quarterly_revenue_usd_millions','quarterly_profit_usd_millions']).copy()
fin_margin['profit_margin_pct'] = (
    fin_margin['quarterly_profit_usd_millions'] /
    fin_margin['quarterly_revenue_usd_millions'] * 100).round(2)

fig, ax = plt.subplots(figsize=(10, 4))
fin_margin.set_index('platform')['profit_margin_pct'].sort_values().plot(
    kind='barh', ax=ax, color=['red' if x < 0 else 'steelblue'
                               for x in fin_margin.set_index('platform')['profit_margin_pct'].sort_values()])
ax.set_title('Zisková marže platforem (%)')
ax.set_xlabel('Zisková marže (%)')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('profit_margin.png', dpi=150)
plt.show()

display(fin_margin[['platform','quarterly_revenue_usd_millions',
                     'quarterly_profit_usd_millions','profit_margin_pct']]
        .sort_values('profit_margin_pct', ascending=False))

## 6. Analýza – streaming_shifts_2026

In [ ]:
print('=== Statistiky streaming_shifts ===')
display(shifts[['popularity','vote_average','vote_count']].describe().round(2))

print('\n=== Distribuce po platformách ===')
plat_counts = pd.Series({
    'Netflix':      int(shifts['on_netflix'].sum()),
    'Hulu':         int(shifts['on_hulu'].sum()),
    'Amazon Prime': int(shifts['on_prime'].sum()),
    'Více platforem': int((shifts[['on_netflix','on_hulu','on_prime']].sum(axis=1) > 1).sum())
})
print(plat_counts.to_string())

In [ ]:
# Distribuce popularity a vote_average
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

shifts['popularity'].hist(bins=20, ax=axes[0], color='coral', edgecolor='white')
axes[0].set_title('Distribuce popularity (streaming_shifts)')
axes[0].set_xlabel('Popularita')
axes[0].set_ylabel('Počet titulů')
axes[0].axvline(shifts['popularity'].mean(), color='red', linestyle='--',
                label=f'Průměr: {shifts["popularity"].mean():.1f}')
axes[0].legend()

shifts['vote_average'].hist(bins=15, ax=axes[1], color='mediumpurple', edgecolor='white')
axes[1].set_title('Distribuce průměrného hodnocení')
axes[1].set_xlabel('Hodnocení (0–10)')
axes[1].set_ylabel('Počet titulů')
axes[1].axvline(shifts['vote_average'].mean(), color='red', linestyle='--',
                label=f'Průměr: {shifts["vote_average"].mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('shifts_distributions.png', dpi=150)
plt.show()

In [ ]:
# Porovnání platforem v streaming_shifts
platform_stats = pd.DataFrame({
    'Netflix':      shifts[shifts['on_netflix']==1][['popularity','vote_average','vote_count']].mean(),
    'Hulu':         shifts[shifts['on_hulu']==1][['popularity','vote_average','vote_count']].mean(),
    'Amazon Prime': shifts[shifts['on_prime']==1][['popularity','vote_average','vote_count']].mean()
}).T.round(2)

print('=== Průměrné metriky podle platformy (2026) ===')
display(platform_stats)

fig, ax = plt.subplots(figsize=(10, 4))
platform_stats[['popularity','vote_average']].plot(kind='bar', ax=ax,
    color=['steelblue','#6ACC65'])
ax.set_title('Průměrná popularita a hodnocení podle platformy (2026)')
ax.set_xlabel('Platforma')
ax.tick_params(axis='x', rotation=0)
ax.legend(['Popularita','Hodnocení (0-10)'])
plt.tight_layout()
plt.savefig('platform_metrics.png', dpi=150)
plt.show()

## 7. Souhrn datové kvality a úpravy dat

In [ ]:
print('=' * 60)
print('SOUHRN DATOVÉ KVALITY')
print('=' * 60)

for df, name in [(netflix, 'netflix_titles'), (financials, 'platform_financials'), (shifts, 'streaming_shifts')]:
    total   = len(df)
    missing = df.isnull().sum().sum()
    dupl    = df.duplicated().sum()
    print(f'\n{name}:')
    print(f'  Celkem řádků:          {total}')
    print(f'  Celkem sloupců:        {df.shape[1]}')
    print(f'  Chybějící hodnoty:     {missing} ({missing/(total*df.shape[1])*100:.1f} % všech buněk)')
    print(f'  Duplicitní řádky:      {dupl}')

print('\n' + '=' * 60)
print('PROVEDENÉ ÚPRAVY DAT')
print('=' * 60)
print('''
netflix_titles:
  - Pole "rating" obsahuje neplatné hodnoty (minuty místo hodnocení)
    → filtrováno při dotazování pomocí validační whitelist
  - Pole "cast" a "director" mají ~30 % null hodnot
    → importováno s --ignoreBlanks, uloženo jako null v MongoDB
  - Pole "date_added" je textový řetězec ("January 1, 2021")
    → parsování roku provedeno v MongoDB agregacích ($split, $toInt)
  - Pole "duration" je textový řetězec ("90 min" / "2 Seasons")
    → parsování čísla provedeno v MongoDB agregacích ($split, $toInt)

platform_financials:
  - Mnoho null hodnot (různé platformy reportují různé metriky)
    → importováno as-is, null hodnoty ošetřeny v dotazech pomocí $ifNull

streaming_shifts:
  - Hodnoty on_netflix/on_hulu/on_prime jsou 0/1 (integer)
    → v MongoDB dotazech porovnáváno s hodnotou 1
  - Pole "streaming_platforms" je textový seznam ("Netflix, Hulu")
    → pro filtrování se používají binární příznaky on_*
''')

## 8. Propojitelnost datasetů

In [ ]:
# Ověření propojitelnosti
netflix_in_financials = netflix[netflix['type'].notna()]['type'].unique()
shifts_platforms = shifts['streaming_platforms'].str.split(', ').explode().str.split(' with ').str[0].str.strip().value_counts()

print('=== Platformy v streaming_shifts ===')
print(shifts_platforms.head(10).to_string())

print('\n=== Platformy v platform_financials ===')
print(financials['platform'].tolist())

print('\n=== Propojení streaming_shifts → platform_financials ===')
netflix_shifts = shifts[shifts['on_netflix'] == 1]
netflix_finance = financials[financials['platform'] == 'Netflix']
print(f'Tituly na Netflixu v 2026: {len(netflix_shifts)}')
print(f'Netflix finanční záznam:   {len(netflix_finance)} (subscribers: {netflix_finance["subscribers_millions"].values[0]}M)')

print('\n=== Průnik titleů netflix_titles ↔ streaming_shifts ===')
common = set(netflix['title'].dropna()) & set(shifts['title'].dropna())
print(f'Společné tituly: {len(common)} (streaming_shifts obsahuje novější tituly z 2026)')

In [ ]:
# Vizualizace objemu dat
fig, ax = plt.subplots(figsize=(8, 4))
data_sizes = pd.Series({
    'netflix_titles\n(8 807 záznamů)': len(netflix),
    'streaming_shifts\n(81 záznamů)':  len(shifts),
    'platform_financials\n(10 záznamů)': len(financials)
})
data_sizes.plot(kind='bar', ax=ax, color=['steelblue','coral','#6ACC65'],
                logy=True)
ax.set_title('Objem dat v jednotlivých kolekcích (log škála)')
ax.set_ylabel('Počet záznamů (log)')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(data_sizes):
    ax.text(i, v * 1.2, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('dataset_sizes.png', dpi=150)
plt.show()